In [1]:
from pyspark.sql.functions import col,when,udf,round,expr,to_date,regexp_replace
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType, LongType
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
import os
import json

In [2]:
spark = SparkSession.builder \
    .appName("SilverStep") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:3.5.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

24/11/15 19:31:18 WARN Utils: Your hostname, fececa-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
24/11/15 19:31:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/fececa/dev-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/fececa/.ivy2/cache
The jars for the packages stored in: /home/fececa/.ivy2/jars
org.apache.spark#spark-avro_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f2c4bb09-670d-47ef-acd9-a1a0d7beefbb;1.0
	confs: [default]
	found org.apache.spark#spark-avro_2.12;3.5.0 in central
	found org.tukaani#xz;1.9 in central
:: resolution report :: resolve 154ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.spark#spark-avro_2.12;3.5.0 from central in [default]
	org.tukaani#xz;1.9 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	--------------------------------------------------------------------

24/11/15 19:31:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

In [4]:
# Verificar o ambiente
if "dev" in current_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [5]:
with open(dir_path + f'/{env}-env/projeto_steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [6]:
schema = StructType([
    StructField("id", LongType(), True),
    StructField("file_date", DateType(), True),
    StructField("img", StringType(), True),
    StructField("name", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("short_description", StringType(), True),
    StructField("discount", LongType(), True),
    StructField("recommended", StringType(), True),
    StructField("minimum", StringType(), True),
    StructField("link", StringType(), True)
])


In [7]:
path = dir_path + f'/{env}-env/datalake/steam/wishlist/bronze'
df = spark.read.format("avro").load(path)

In [8]:
df = df.select("appid", "file_date", "header_image", "name", "price",
            "short_description","discount_pct","recommended",
            "minimum")

In [9]:
df = df.withColumnRenamed("discount_pct", "discount")\
    .withColumnRenamed("header_image", "img")

In [10]:
df = df.withColumn("price",regexp_replace(col("price"),"R\\$",""))
df = df.withColumn("price",regexp_replace(col("price"),"\\.",""))
df = df.withColumn("price",regexp_replace(col("price"),",","."))
df = df.withColumn("price", round(col("price").cast("float"), 2))

df = df.withColumn("appid", col("appid").cast("long"))

In [11]:
def extrair_texto_html(html):
    if html is None:
        return ""
    soup = BeautifulSoup(html, 'html.parser')
    texto = soup.get_text()
    return texto

extrair_texto_html_udf = udf(extrair_texto_html, StringType())

In [12]:
df = df.withColumn('minimum', when((df['minimum'].isNull()) | (df['minimum'] == ""), "-").otherwise(extrair_texto_html_udf(df['minimum'])))
df = df.withColumn('recommended', when((df['recommended'].isNull()) | (df['recommended'] == ""), "-").otherwise(extrair_texto_html_udf(df['recommended'])))

In [13]:
df = df.withColumn('minimum', expr("regexp_replace(minimum, 'Minimum:', '')"))
df = df.withColumn('minimum', expr("regexp_replace(minimum, 'Minimum', '')"))
df = df.withColumn('recommended', expr("regexp_replace(recommended, 'Recommended:', '')"))
df = df.withColumn('recommended', expr("regexp_replace(recommended, 'Recommended', '')"))

In [14]:
df = df.withColumn("link", F.concat(F.lit("https://store.steampowered.com/app/"), F.col("appid")))

In [15]:
df = df.withColumn("file_date", to_date(df["file_date"], "yyyy-MM-dd"))

In [16]:
df = spark.createDataFrame(df.rdd, schema)

In [17]:
string_cols = [f.name for f in df.schema.fields if f.dataType == StringType()]
for col_name in string_cols:
    df = df.withColumn(col_name, when(col(col_name).isNull(), '-').otherwise(col(col_name)))
    df = df.withColumn(col_name, when(col(col_name) == 'nan', '-').otherwise(col(col_name)))
    df = df.withColumn(col_name, when(col(col_name) == 'N/A', '-').otherwise(col(col_name)))

In [18]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- file_date: date (nullable = true)
 |-- img: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: float (nullable = true)
 |-- short_description: string (nullable = true)
 |-- discount: long (nullable = true)
 |-- recommended: string (nullable = true)
 |-- minimum: string (nullable = true)
 |-- link: string (nullable = true)



In [19]:
df.show()

/tmp/ipykernel_7459/2140015686.py:4: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.


+-------+----------+--------------------+--------------------+------+--------------------+--------+--------------------+--------------------+--------------------+
|     id| file_date|                 img|                name| price|   short_description|discount|         recommended|             minimum|                link|
+-------+----------+--------------------+--------------------+------+--------------------+--------+--------------------+--------------------+--------------------+
|1466860|2024-11-15|https://shared.ak...|Age of Empires IV...| 99.99|Celebrating its f...|       0|Requires a 64-bit...|Requires a 64-bit...|https://store.ste...|
|  15100|2024-11-15|https://shared.ak...|Assassin's Creed™...| 59.99|Assassin's Creed™...|       0|                   -|  Supported OS: W...|https://store.ste...|
| 201870|2024-11-15|https://shared.ak...|Assassin's Creed®...| 59.99|Ezio Auditore wal...|       0| OS *: Windows® X...| OS *: Windows® X...|https://store.ste...|
| 242050|2024-11-15|ht

In [20]:
output_path = dir_path + f'/{env}-env/datalake/steam/wishlist/silver'
os.makedirs(output_path, exist_ok=True)

In [21]:
df.write.mode('overwrite').partitionBy('file_date').parquet(output_path)

/tmp/ipykernel_7459/2140015686.py:4: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
